# 🔀 Git & GitHub Actions — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Each topic includes: real-world scenarios from production companies, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---

**Topics**
1. Git Internals — What Git Actually Is
2. Branching Strategies at Scale
3. The CI/CD Pipeline Mental Model
4. GitHub Actions Architecture
5. Secrets & Security in Pipelines
6. Advanced Workflow Patterns
7. Monorepo vs Polyrepo
8. Real-World Case Studies (Netflix, Shopify, Google)
9. The DevOps Architect's Pipeline Design Framework

---
## 1 · Git Internals — What Git Actually Is

### 🧠 Mental Model — *The Content-Addressed Filesystem*

> **Git is not a "track changes" tool. It is a content-addressed, immutable, append-only directed acyclic graph (DAG) of snapshots. Every commit is a SHA-256 hash of the ENTIRE repository state at that moment. This design makes tampering physically detectable.**

**WHY it exists:** Before Git (2005), teams used SVN (centralized — single server = single point of failure) or CVS (even older, file-level locking). Linus Torvalds built Git because BitKeeper (then used for Linux kernel) revoked their license. The core insight: **distributed version control means every clone IS the backup**.

**WHAT it is:** A key-value store where:
- **Key** = SHA-1 hash of the content (40 hex chars)
- **Value** = one of four object types: `blob` (file), `tree` (directory), `commit`, `tag`

**HOW it works — the object graph:**

```
commit a3f4b1c
├── parent: 9e2d3f8  (previous commit)
├── author / timestamp
├── message
└── tree 7a2c9d1  ← points to root directory snapshot
         ├── blob 4f3a2b1  ← src/main.py
         ├── blob 9c8d7e2  ← README.md
         └── tree 2b4e6f8  ← src/utils/
                  └── blob 1a5c3d9  ← src/utils/helpers.py
```

**Key insight:** If any file changes, its blob hash changes, which changes the tree hash, which changes the commit hash. **You cannot alter history silently.** This is why `git push --force` is dangerous on shared branches — it rewrites history that others have already built on.

### The Three Trees (the hardest Git concept for most engineers)

| Name | What it stores | How you interact |
|---|---|---|
| **HEAD** | The last commit (or current branch pointer) | `git reset --hard HEAD` |
| **Index (Staging Area)** | What will go into the NEXT commit | `git add`, `git diff --staged` |
| **Working Directory** | Files on disk | Edit files, `git diff` |

```
Working Directory  →  git add  →  Index (Staging)  →  git commit  →  HEAD
       ↑                                                                  |
       └───────────────── git checkout / git restore ←───────────────────┘
```

### 🌍 Real-World: How Git's Design Prevents Disasters at Google Scale
- **Google's monorepo (Piper):** 2+ billion lines of code. The DAG structure means a single corrupted node (file) is immediately detectable because the SHA chain breaks.
- **Linux kernel:** 1,000+ contributors submitting patches. The SHA integrity means Linus can accept a patch from an unknown contributor knowing that the bytes he received are exactly what was sent — no tampering in transit.
- **GitHub's object storage:** Every repository on GitHub is the same Git object store, deduplicated by SHA. Two repos with the same file content share the same blob object.

In [ ]:
"""
Git Internals Simulator
=======================
This code replicates Git's core object model: content-addressed storage
where the key is the SHA of the content.

Run this to understand WHY git blame, git bisect, and git log are so fast:
they are traversals of an immutable DAG with O(1) hash lookups.
"""
from __future__ import annotations
import hashlib
import json
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

In [ ]:
class ObjectStore:
    """Git's .git/objects — a content-addressed key-value store."""
    def __init__(self):
        self._store: dict[str, dict] = {}

    def write(self, obj: dict) -> str:
        """Store object, return its SHA (the 'key' in Git's KV store)."""
        content = json.dumps(obj, sort_keys=True).encode()
        sha = hashlib.sha1(content).hexdigest()  # Git uses SHA-1
        self._store[sha] = obj
        return sha

    def read(self, sha: str) -> dict:
        return self._store[sha]

    def exists(self, sha: str) -> bool:
        return sha in self._store


store = ObjectStore()


def make_blob(content: str) -> str:
    """A blob is just file content. Its identity IS its content."""
    return store.write({"type": "blob", "content": content})


def make_tree(entries: dict[str, str]) -> str:
    """A tree maps filename -> blob/tree SHA. A directory snapshot."""
    return store.write({"type": "tree", "entries": entries})


def make_commit(tree_sha: str, message: str, parent_sha: Optional[str] = None) -> str:
    """A commit points to a tree + optional parent commit."""
    return store.write({
        "type": "commit",
        "tree": tree_sha,
        "parent": parent_sha,
        "message": message,
        "timestamp": datetime.now().isoformat(),
    })

In [ ]:
# Simulate: engineer creates a file, commits it, then modifies it

# === Commit 1: Initial state ===
main_py_v1 = make_blob("print('hello world')")
readme_v1  = make_blob("# My Project")
tree_v1    = make_tree({"main.py": main_py_v1, "README.md": readme_v1})
commit_1   = make_commit(tree_v1, "Initial commit")

# === Commit 2: Only README changes ===
readme_v2  = make_blob("# My Project\n\nA production service.")  # changed
tree_v2    = make_tree({"main.py": main_py_v1, "README.md": readme_v2})  # main.py REUSED
commit_2   = make_commit(tree_v2, "Update README", parent_sha=commit_1)

print("=== Git Object Store Demo ===")
print(f"\nCommit 1 SHA:    {commit_1}")
print(f"Commit 2 SHA:    {commit_2}")
print(f"\nmain.py blob is IDENTICAL across both commits:")
print(f"  tree_v1['main.py'] == tree_v2['main.py']: ", end="")
print(store.read(tree_v1)["entries"]["main.py"] == store.read(tree_v2)["entries"]["main.py"])

print(f"\nREADME changed, so its blob SHA is different:")
print(f"  v1 README SHA: {readme_v1[:16]}...")
print(f"  v2 README SHA: {readme_v2[:16]}...")

print(f"\n✅ KEY INSIGHT: Unchanged files are DEDUPLICATED by SHA.")
print(f"   Git does not store file diffs — it stores SNAPSHOTS, but deduplicates via content-addressing.")
print(f"   Total objects in store: {len(store._store)}")

In [ ]:
# Demonstrate: altering a commit's content breaks the hash chain
# This is WHY 'git push --force' on shared branches is catastrophic

def traverse_history(head_sha: str) -> list[dict]:
    """Walk the commit DAG from HEAD backwards — this is git log."""
    history = []
    current = head_sha
    while current:
        obj = store.read(current)
        history.append({"sha": current[:8], "message": obj["message"]})
        current = obj.get("parent")
    return history


print("=== git log (DAG traversal from HEAD) ===")
for entry in traverse_history(commit_2):
    print(f"  commit {entry['sha']}  {entry['message']}")

print("\n=== The Branch Model ===")
print("A branch is just a POINTER (40 bytes) to a commit SHA.")
print("Creating 1,000 branches costs almost nothing.")

branches = {
    "main":               commit_2,
    "feature/dark-mode":  commit_2,  # branch from same SHA = zero cost
    "hotfix/login-crash": commit_1,
}

for branch, sha in branches.items():
    print(f"  refs/heads/{branch} → {sha[:8]}")

---
## 2 · Branching Strategies at Scale

### 🧠 Mental Model — *The Blast Radius Dial*

> **Every branching strategy is a trade-off between ISOLATION (protection from others' chaos) and INTEGRATION SPEED (how quickly your work reaches shared state). The more isolation, the more merge pain. The less isolation, the more you need automated quality gates.**

**WHY:** Teams working on the same codebase simultaneously need a protocol for combining their work. Without one, you get "integration hell" — days before a release trying to merge months of parallel work.

**WHAT:** A branching strategy defines:
1. Which branches are long-lived (main, develop, release)
2. When you branch (start of feature? start of release?)
3. How you integrate (merge commit? squash? rebase?)
4. Who can merge where (branch protection rules)

### Strategy Comparison with Visual Flow

```mermaid
gitGraph
   commit id: "initial"
   branch develop
   checkout develop
   commit id: "feat: login"
   branch feature/payments
   checkout feature/payments
   commit id: "add stripe"
   commit id: "add webhook"
   checkout develop
   merge feature/payments id: "merge payments"
   branch release/1.0
   checkout release/1.0
   commit id: "bump version"
   checkout main
   merge release/1.0 id: "v1.0" tag: "v1.0.0"
```

### The Decision Matrix

| Strategy | Release Frequency | Team Size | Multiple Versions? | Complexity |
|---|---|---|---|---|
| **Trunk-Based** | Many/day | Any (with discipline) | No | Low |
| **GitHub Flow** | Daily | Small-Medium | No | Low |
| **GitFlow** | Weekly/monthly | Medium-Large | Yes | High |
| **Release Flow** (MS) | Per sprint | Large | Yes | Medium |
| **Ship/Show/Ask** | Continuous | Senior teams | No | Low |

### 🌍 Real-World: How Companies Choose
- **Netflix:** Trunk-based. Engineers merge to main multiple times per day. Feature flags (via LaunchDarkly) gate incomplete features. Canary deployments ensure safe rollout.
- **Shopify:** GitHub Flow (short feature branches → PR → squash merge to main → auto-deploy). Their deployment pipeline runs in < 10 minutes.
- **Microsoft Windows team:** Release Flow — branches from `main` into `release/sprint-N`, cherry-picks for hotfixes. Different parts of Windows have different release cadences.
- **Linux kernel:** Hierarchical — Linus maintains `main`, subsystem maintainers maintain their trees, patches flow up through the hierarchy via pull requests to maintainers.

### ⚠️ The Merge Hell Anti-Pattern

```
❌ BEFORE (GitFlow misused by a CD team):
feature/A lives 3 weeks ──────────────────────────────────────────────┐
feature/B lives 3 weeks ──────────────────────────────────────────────┤
                                                                        ↓ merge conflict nightmare
develop ───────────────────────────────────────────────────────── MERGE WAR

✅ AFTER (Trunk-Based with feature flags):
main ──── commit ── commit ── commit ── commit ── commit ── commit ──▶
          (A-wip)   (B-wip)   (A-wip)   (B-done)  (A-done)  (deploy)
          flag OFF  flag OFF   flag OFF  flag ON   flag ON
```

In [ ]:
"""
Branching Strategy Simulator
============================
Models different branching strategies and shows the merge complexity cost
of each approach. This is the quantitative argument for trunk-based dev.
"""
import random
from dataclasses import dataclass
from typing import List


@dataclass
class Branch:
    name: str
    diverged_commits: int  # how many commits since branching from main
    age_days: int

    @property
    def merge_conflict_probability(self) -> float:
        """Empirically, conflict probability grows with branch age and commit count."""
        # Simplified model: P(conflict) ≈ 1 - e^(-k * diverged_commits)
        import math
        k = 0.05  # conflict rate per diverged commit
        return 1 - math.exp(-k * self.diverged_commits)

    @property
    def integration_cost_hours(self) -> float:
        """Estimated engineer-hours to integrate, based on branch age."""
        return 0.5 + (self.age_days * 0.3) + (self.diverged_commits * 0.1)


def simulate_gitflow(num_features: int, branch_age_days: int = 14) -> dict:
    """GitFlow: features live as long branches before merging."""
    branches = [
        Branch(
            name=f"feature/{i}",
            diverged_commits=random.randint(10, 50),
            age_days=branch_age_days,
        )
        for i in range(num_features)
    ]
    total_cost = sum(b.integration_cost_hours for b in branches)
    avg_conflict_p = sum(b.merge_conflict_probability for b in branches) / len(branches)
    return {"strategy": "GitFlow", "total_cost_hours": total_cost, "avg_conflict_probability": avg_conflict_p}


def simulate_trunk_based(num_features: int, branch_age_days: int = 1) -> dict:
    """Trunk-based: branches live < 1 day, tiny divergence."""
    branches = [
        Branch(
            name=f"feature/{i}",
            diverged_commits=random.randint(1, 5),  # much smaller
            age_days=branch_age_days,
        )
        for i in range(num_features)
    ]
    total_cost = sum(b.integration_cost_hours for b in branches)
    avg_conflict_p = sum(b.merge_conflict_probability for b in branches) / len(branches)
    return {"strategy": "Trunk-Based", "total_cost_hours": total_cost, "avg_conflict_probability": avg_conflict_p}


random.seed(42)
print("=== Branching Strategy Cost Model (10 features, team of 5) ===")
gitflow_result = simulate_gitflow(num_features=10)
trunk_result   = simulate_trunk_based(num_features=10)

for result in [gitflow_result, trunk_result]:
    print(f"\n{result['strategy']}:")
    print(f"  Integration cost:       {result['total_cost_hours']:.1f} engineer-hours")
    print(f"  Avg conflict P per PR:  {result['avg_conflict_probability']:.1%}")

---
## 3 · The CI/CD Pipeline Mental Model

### 🧠 Mental Model — *The Quality Gate Assembly Line*

> **A CI/CD pipeline is a quality gate assembly line. Each station (job) checks a specific failure mode. If a check fails, the piece is rejected before reaching the next stage. The car doesn't reach the customer with a broken engine because it was caught at the engine test station — not at the dealer.**

**WHY CI/CD exists:** Before automated pipelines, the sequence was: write code → manually run tests (if you remembered) → manually build → manually copy files to server → hope it works. Each step was error-prone, slow, and non-reproducible. "It works on my machine" was the universal excuse.

**WHAT it is:** An automated system that takes a code change from developer laptop to production, with every step verified by machines.

**HOW it maps to value:**

| Stage | What it catches | If skipped |
|---|---|---|
| Linting | Style violations, obvious bugs | Code review time wasted on style |
| Unit tests | Logic errors in individual functions | Bugs shipped to integration |
| Build | Compilation/import errors | Deployment fails at runtime |
| Integration tests | Component interaction bugs | Works in unit, breaks in system |
| Security scan | CVEs, secrets, SAST | Vulnerabilities shipped to prod |
| Staging deploy | Environment-specific issues | Works in CI, breaks in prod |
| Smoke tests | Critical path broken | Users hit broken homepage |
| Production deploy | — | Manual work, inconsistency |

### The DORA Metrics — How You Measure Pipeline Health

Google's DevOps Research and Assessment (DORA) team identified four metrics that predict software delivery performance:

| Metric | Elite | High | Medium | Low |
|---|---|---|---|---|
| **Deployment frequency** | Multiple/day | Weekly | Monthly | 6+ months |
| **Lead time for changes** | < 1 hour | 1 day-1 week | 1-6 months | 6+ months |
| **Change failure rate** | 0-5% | 5-10% | 10-15% | > 15% |
| **Recovery time** | < 1 hour | < 1 day | 1 day-1 week | > 1 week |

> **Your pipeline is the primary lever for improving all four metrics.** Fast pipelines → high deployment frequency. Good tests → low change failure rate. Automated rollback → fast recovery time.

### 🌍 Real-World: Amazon's Deployment Philosophy
Amazon deploys to production every **11.6 seconds** on average. This is only possible because:
1. Every deployment is small (small batches = small blast radius)
2. Automated rollback triggers on metric degradation
3. Each service team owns their pipeline — no centralized release gate
4. "You build it, you run it" — engineers are on-call for their own services

### The Pipeline Anti-Patterns

```
❌ ANTI-PATTERN 1: The Fragile Snowflake
   Pipeline that only one person understands and can fix.
   Sign: "Ask Jenkins Dave, he knows how to unblock it."
   Fix: Document every step. Use reusable workflows. Automate cleanup.

❌ ANTI-PATTERN 2: The Re-Run Culture
   "Just re-run it, it's flaky" becomes the answer to every failure.
   Sign: Team re-runs pipelines without investigating why they failed.
   Fix: Fix flaky tests. They are technical debt that compounds.

❌ ANTI-PATTERN 3: The Monolithic Pipeline
   One giant sequential pipeline that takes 90 minutes.
   Sign: Engineers push and go to lunch.
   Fix: Parallelize. Gate on fast checks first. Defer slow tests.

❌ ANTI-PATTERN 4: Deploy = Copy Files
   Deployment is an SSH + rsync script.
   Sign: Can't roll back. Can't reproduce old deploy. Snowflake servers.
   Fix: Immutable artifacts (Docker images). Declarative deployment (K8s, Terraform).
```

---
## 4 · GitHub Actions Architecture

### 🧠 Mental Model — *The Event-Driven Automation Platform*

> **GitHub Actions is an event-driven workflow engine built into your source control. An event (push, PR, schedule) triggers a workflow, which runs jobs on runners (VMs), which execute steps (commands or actions). The key architectural insight is that the workflow YAML is code — version-controlled, reviewable, testable.**

### Component Hierarchy

```
Repository
└── Workflow (.github/workflows/ci.yml)
    ├── Trigger (on: push, pull_request, schedule...)
    └── Jobs (parallel by default)
        ├── Job: test
        │   ├── runs-on: ubuntu-latest (Runner VM)
        │   └── Steps (sequential within a job)
        │       ├── Step: actions/checkout@v4
        │       ├── Step: Setup Python
        │       └── Step: Run tests
        └── Job: build
            ├── needs: [test]  ← dependency between jobs
            └── Steps...
```

### Runners: The Execution Environment

```
GitHub-Hosted Runners:
┌─────────────────────────────────────────────────────────────┐
│  ubuntu-latest  │  windows-latest  │  macos-latest          │
│  (2-core, 7GB)  │  (2-core, 7GB)   │  (3-core, 14GB M1)     │
│  Ephemeral VM   │  Ephemeral VM    │  Ephemeral VM           │
│  Free tier: 2000 min/month (public repos: unlimited)         │
└─────────────────────────────────────────────────────────────┘

Self-Hosted Runners:
┌─────────────────────────────────────────────────────────────┐
│  Your VMs / K8s pods / physical machines                     │
│  Use when: private network access, GPU, compliance           │
│  Risk: You manage updates, security, scaling                 │
└─────────────────────────────────────────────────────────────┘
```

### Actions: The Reusable Building Blocks

| Action Type | Example | What it is |
|---|---|---|
| **Docker action** | `actions/checkout@v4` | Runs a container — portable, isolated |
| **JavaScript action** | Most marketplace actions | Node.js script — fast startup, no Docker needed |
| **Composite action** | Your custom `./actions/notify` | YAML-defined sequence of steps — DRY reuse |

**Where this is seen in frameworks:**
- `actions/setup-python` — sets up Python and caches pip
- `actions/cache` — caches arbitrary directories keyed by lockfile hash
- `docker/build-push-action` — builds and pushes Docker images
- `aws-actions/configure-aws-credentials` — OIDC-based AWS auth
- `hashicorp/setup-terraform` — installs and configures Terraform

### The Trigger System — Event-Driven Architecture in Practice

```yaml
# GitHub Actions triggers map to the same event-driven patterns
# used in distributed systems (Kafka topics, SQS queues)

on:
  push:
    branches: [main]           # Source: main branch pushes
    paths: ["src/**", "*.py"]  # Filter: only code changes (not docs)
    
  pull_request:
    types: [opened, synchronize, reopened]  # PR lifecycle events
    
  schedule:
    - cron: "0 2 * * 1"        # Monday 2AM UTC — nightly security scans
    
  workflow_dispatch:            # Manual trigger with inputs
    inputs:
      environment:
        description: "Deploy to"
        required: true
        type: choice
        options: [staging, production]
        
  workflow_call:                # Called by another workflow (reusable)
    inputs:
      image-tag:
        type: string
        required: true
```

### Contexts: The Runtime Data Available to Your Workflow

| Context | Example | When to use |
|---|---|---|
| `github.sha` | `a3f4b1c9...` | Pin Docker image to exact commit |
| `github.ref_name` | `main`, `v1.2.3` | Conditionally deploy on tag |
| `github.event_name` | `push`, `pull_request` | Different behavior per trigger |
| `github.actor` | `karthik-dev` | Audit who triggered a deployment |
| `runner.os` | `Linux` | Cross-platform workflows |
| `secrets.MY_SECRET` | `***` | Injected at runtime, never logged |

In [ ]:
"""
GitHub Actions Workflow Design Patterns
=======================================
This module generates well-structured GitHub Actions workflow YAML
and validates it against best practices.

Think of this as 'Infrastructure as Code' for your CI/CD.
"""
from __future__ import annotations
import textwrap
from dataclasses import dataclass, field
from typing import Any
import yaml  # pip install pyyaml

In [ ]:
# ============================================================
# ❌ BEFORE: A naive workflow that violates best practices
# ============================================================

naive_workflow = """
# ❌ PROBLEMS:
#   1. No dependency caching (reinstalls packages every run)
#   2. No job parallelism (all steps sequential)
#   3. Runs on every push to every branch (noisy)
#   4. No timeout (can hang forever and consume minutes)
#   5. Uses 'latest' tag for actions (supply chain risk)
#   6. Secrets exposed via echo (security issue)

name: CI
on: push

jobs:
  ci:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@main          # ❌ not pinned to SHA
      - run: pip install -r requirements.txt  # ❌ no cache
      - run: flake8 .                         # ❌ sequential
      - run: pytest tests/                    # ❌ sequential
      - run: pytest integration/              # ❌ sequential
      - run: echo ${{ secrets.API_KEY }}      # ❌ SECRET IN LOGS!
"""

print("❌ NAIVE WORKFLOW PROBLEMS:")
for line in naive_workflow.strip().split('\n'):
    if '❌' in line:
        print(f"  {line.strip()}")

In [ ]:
# ============================================================
# ✅ AFTER: Production-grade workflow with all best practices
# ============================================================

PRODUCTION_WORKFLOW = """
name: CI/CD Pipeline

on:
  push:
    branches: [main]
    paths:
      - 'src/**'
      - 'tests/**'
      - 'requirements*.txt'
      - '.github/workflows/ci.yml'
  pull_request:
    branches: [main]
    types: [opened, synchronize, reopened]

# Cancel in-progress runs on new push (saves runner minutes)
concurrency:
  group: ${{ github.workflow }}-${{ github.ref }}
  cancel-in-progress: ${{ github.ref != 'refs/heads/main' }}

env:
  PYTHON_VERSION: '3.11'
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}

jobs:
  # ─── Gate 1: Fast feedback (runs in parallel) ────────────────────────
  lint:
    name: Lint & Format
    runs-on: ubuntu-latest
    timeout-minutes: 5  # ✅ never hang
    steps:
      - uses: actions/checkout@v4  # ✅ pinned major version
      
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip  # ✅ caches pip packages by lockfile hash
          
      - run: pip install ruff
      - run: ruff check .
      - run: ruff format --check .

  unit-tests:
    name: Unit Tests
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
          
      - run: pip install -r requirements-dev.txt
      - run: pytest tests/unit/ -v --tb=short --cov=src --cov-report=xml
      
      - uses: codecov/codecov-action@v4
        with:
          token: ${{ secrets.CODECOV_TOKEN }}  # ✅ secret, never echoed
          fail_ci_if_error: false  # coverage drop = warning, not failure

  security-scan:
    name: Security Scan
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      
      - name: Dependency vulnerability scan
        uses: pypa/gh-action-pip-audit@v1.1.0
        with:
          inputs: requirements.txt
          
      - name: SAST scan
        uses: returntocorp/semgrep-action@v1
        with:
          config: p/python p/security-audit

  # ─── Gate 2: Build (only after Gate 1 passes) ─────────────────────
  build:
    name: Build Docker Image
    runs-on: ubuntu-latest
    needs: [lint, unit-tests, security-scan]  # ✅ all Gate 1 must pass
    timeout-minutes: 20
    permissions:
      contents: read
      packages: write
      id-token: write  # ✅ OIDC token for registry auth
    outputs:
      image-tag: ${{ steps.meta.outputs.tags }}
    steps:
      - uses: actions/checkout@v4
      
      - uses: docker/setup-buildx-action@v3
      
      - uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}  # Auto-generated, no rotation needed
          
      - id: meta
        uses: docker/metadata-action@v5
        with:
          images: ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}
          tags: |
            type=sha,prefix=sha-
            type=ref,event=branch
            type=semver,pattern={{version}}
            
      - uses: docker/build-push-action@v6
        with:
          context: .
          push: ${{ github.event_name != 'pull_request' }}
          tags: ${{ steps.meta.outputs.tags }}
          cache-from: type=gha  # ✅ GitHub Actions cache for Docker layers
          cache-to: type=gha,mode=max

  # ─── Gate 3: Deploy to staging, then prod ─────────────────────────
  deploy-staging:
    name: Deploy → Staging
    runs-on: ubuntu-latest
    needs: [build]
    if: github.ref == 'refs/heads/main'  # Only on main branch
    environment: staging  # ✅ requires environment protection rules
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to staging cluster
        run: |
          echo "Deploying ${{ needs.build.outputs.image-tag }} to staging"
          # kubectl set image deployment/app app=${{ needs.build.outputs.image-tag }}

  deploy-production:
    name: Deploy → Production
    runs-on: ubuntu-latest
    needs: [deploy-staging]
    if: github.ref == 'refs/heads/main'
    environment:
      name: production  # ✅ requires manual approval if configured
      url: https://myapp.com
    timeout-minutes: 15
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to production cluster
        run: echo "Deploying to production"
"""

print("✅ PRODUCTION WORKFLOW DESIGN PRINCIPLES:")
principles = [
    "Path-filtered triggers (only runs on relevant changes)",
    "Concurrency groups (cancels stale runs, queues prod deploys)",
    "Parallel Gate 1 jobs (lint + tests + security run simultaneously)",
    "Gate 2 (build) only starts when ALL Gate 1 jobs pass",
    "Dependency caching (pip cache keyed by requirements.txt hash)",
    "Timeout on every job (never hang, fail fast)",
    "GITHUB_TOKEN for registry auth (no stored credentials)",
    "Docker layer caching via GitHub Actions cache",
    "Environment protection rules (manual approval for production)",
    "Job outputs (image tag flows from build → deploy jobs)",
]
for i, p in enumerate(principles, 1):
    print(f"  {i:02d}. {p}")

print(f"\nWorkflow YAML length: {len(PRODUCTION_WORKFLOW)} chars")
print("Save to: .github/workflows/ci.yml")

---
## 5 · Secrets & Security in Pipelines

### 🧠 Mental Model — *The Least-Privilege Principle Applied to Automation*

> **Every secret in a pipeline is a liability. The goal of a senior DevOps architect is to eliminate secrets, not rotate them. OIDC is the mechanism. The principle is: no credential should outlive the job that needs it.**

### The Secrets Attack Surface

```
Where secrets leak in pipelines:

1. YAML files committed to repo          → Detected by secret scanning
2. Log output (echo $SECRET)             → Masked by ::add-mask::
3. Environment variable in child process → Use secrets context, not env
4. Artifact uploads containing .env      → .gitignore + artifact exclusion
5. PR from fork triggering workflow      → fork PRs can't access secrets by default
6. Compromised Action in marketplace     → Pin to SHA, not version tag
7. Long-lived credentials = long risk    → Replace with OIDC
```

### OIDC: The Architecture That Eliminates Long-Lived Credentials

```mermaid
sequenceDiagram
    participant GH as GitHub Actions Job
    participant GHOIDC as GitHub OIDC Provider
    participant AWS as AWS STS
    participant S3 as AWS S3

    GH->>GHOIDC: Request OIDC JWT token
    GHOIDC-->>GH: Short-lived JWT (expires: job end)
    GH->>AWS: AssumeRoleWithWebIdentity(JWT, role-arn)
    AWS->>GHOIDC: Verify JWT signature
    GHOIDC-->>AWS: Valid (from repo: org/repo, branch: main)
    AWS-->>GH: Temporary credentials (15min TTL)
    GH->>S3: Upload artifact (using temp credentials)
    Note over GH,S3: No AWS keys stored anywhere!
```

### Security Scanning Layers

| Layer | Tool | What it finds | When to run |
|---|---|---|---|
| **Secrets in code** | trufflesecurity/trufflehog | Credentials, API keys | Pre-commit + PR |
| **Dependency CVEs** | pip-audit, npm audit, trivy | Known CVEs in deps | Every PR |
| **SAST** | Semgrep, Bandit | Insecure code patterns | Every PR |
| **Container vulns** | Trivy, Grype | OS + app CVEs in image | Post-build |
| **IaC misconfig** | tfsec, checkov | Terraform/K8s issues | IaC PRs |
| **Runtime** | Falco, Sysdig | Anomalous behavior | Production |

### 🌍 Real-World: The Codecov Supply Chain Attack (2021)
Attackers modified Codecov's bash uploader script (used in CI pipelines). Any
team that ran `curl | bash` to install Codecov had their `$CODECOV_TOKEN` stolen.
Thousands of companies were affected, including Twitch, Twilio, and HashiCorp.

**What should have prevented it:**
- Pinning the script to a known SHA
- Not using `curl | bash` (pipe to bash is inherently unsafe)
- Using the official GitHub Action (`codecov/codecov-action@v4`) which is reviewed and signed
- Minimal permissions: the token should only have permission to upload coverage, not read secrets

**Lesson:** Third-party Actions and scripts are transitive dependencies. Treat them with the same scrutiny you'd give a PyPI package.

In [ ]:
"""
Secret Security Auditor
=======================
Scans workflow YAML files for common security anti-patterns.
This is the kind of tool a DevOps architect builds to enforce
standards across an organization's pipelines.
"""
from __future__ import annotations
import re
from dataclasses import dataclass
from enum import Enum
from typing import List


class Severity(Enum):
    CRITICAL = "CRITICAL"
    HIGH     = "HIGH"
    MEDIUM   = "MEDIUM"
    INFO     = "INFO"


@dataclass
class Finding:
    rule: str
    severity: Severity
    line: int
    message: str
    fix: str


class WorkflowAuditor:
    """Static analysis for GitHub Actions workflow security."""

    RULES = [
        {
            "name": "SECRET_IN_ECHO",
            "pattern": r"echo.*\${{\s*secrets\.",
            "severity": Severity.CRITICAL,
            "message": "Secret value echoed to logs — will appear in job output",
            "fix": "Remove the echo. Use 'add-mask' if you need to verify a secret is set.",
        },
        {
            "name": "UNPINNED_ACTION",
            "pattern": r"uses:\s+[^@]+@(main|master|latest|v\d+\.\d+)",
            "severity": Severity.HIGH,
            "message": "Action pinned to mutable ref — vulnerable to supply chain attack",
            "fix": "Pin to SHA: 'uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683'",
        },
        {
            "name": "NO_TIMEOUT",
            "pattern": r"^\s+[a-z-]+:\s*$",  # job definition without timeout
            "severity": Severity.MEDIUM,
            "message": "Job has no timeout-minutes — can run indefinitely and block runners",
            "fix": "Add 'timeout-minutes: N' to every job.",
        },
        {
            "name": "CURL_PIPE_BASH",
            "pattern": r"curl.+\|.*bash",
            "severity": Severity.CRITICAL,
            "message": "curl | bash — remote code execution if the script URL is compromised",
            "fix": "Download the script, verify its SHA, then run it. Or use an official Action.",
        },
        {
            "name": "HARDCODED_SECRET",
            "pattern": r"(password|api[_-]?key|token|secret):\s+['\"][^$\s]{8,}",
            "severity": Severity.CRITICAL,
            "message": "Hardcoded credential value in workflow YAML",
            "fix": "Move to GitHub Secrets: ${{ secrets.MY_SECRET }}",
        },
    ]

    def audit(self, workflow_yaml: str) -> List[Finding]:
        findings = []
        lines = workflow_yaml.split("\n")
        for lineno, line in enumerate(lines, 1):
            for rule in self.RULES:
                if re.search(rule["pattern"], line, re.IGNORECASE):
                    findings.append(Finding(
                        rule=rule["name"],
                        severity=rule["severity"],
                        line=lineno,
                        message=rule["message"],
                        fix=rule["fix"],
                    ))
        return findings


# Test against the naive workflow
auditor = WorkflowAuditor()
findings = auditor.audit(naive_workflow)

print("=== Workflow Security Audit Results ===")
print(f"Found {len(findings)} issue(s)\n")
for f in findings:
    print(f"[{f.severity.value}] Rule: {f.rule} (line {f.line})")
    print(f"  Issue: {f.message}")
    print(f"  Fix:   {f.fix}")
    print()

---
## 6 · Advanced Workflow Patterns

### 🧠 Mental Model — *DRY Pipelines Through Composition*

> **A 200-engineer org with 50 microservices should NOT have 50 copies of the same CI workflow. Reusable workflows and composite actions are the DRY principle applied to pipeline code. The same way you extract a function to avoid repetition, you extract a reusable workflow.**

### Pattern 1: Reusable Workflows

```yaml
# .github/workflows/_deploy.yml  (reusable, prefixed with _ by convention)
on:
  workflow_call:
    inputs:
      environment:
        type: string
        required: true
      image-tag:
        type: string
        required: true
    secrets:
      KUBE_CONFIG:
        required: true

jobs:
  deploy:
    runs-on: ubuntu-latest
    environment: ${{ inputs.environment }}
    steps:
      - name: Deploy ${{ inputs.image-tag }} to ${{ inputs.environment }}
        run: |
          echo ${{ secrets.KUBE_CONFIG }} | base64 -d > kubeconfig
          kubectl set image deployment/app app=${{ inputs.image-tag }}
          kubectl rollout status deployment/app
```

```yaml
# Used by 50 different service workflows:
jobs:
  deploy-staging:
    uses: ./.github/workflows/_deploy.yml
    with:
      environment: staging
      image-tag: ${{ needs.build.outputs.image-tag }}
    secrets: inherit
```

### Pattern 2: Matrix Builds (Testing Across Versions)

```yaml
# Tests against Python 3.9, 3.10, 3.11, 3.12 simultaneously
jobs:
  test:
    strategy:
      fail-fast: false  # all matrix jobs run even if one fails
      matrix:
        python-version: ["3.9", "3.10", "3.11", "3.12"]
        os: [ubuntu-latest, windows-latest, macos-latest]
    runs-on: ${{ matrix.os }}
    steps:
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pytest tests/
```

### Pattern 3: Deployment Gates with Manual Approval

```yaml
# Environment 'production' is configured in GitHub with:
#   - Required reviewers: [lead-architect, security-team]
#   - Wait timer: 5 minutes (canary observation period)
#   - Deployment branches: main only

deploy-production:
  environment:
    name: production
    url: https://api.myapp.com
  # GitHub pauses here and emails required reviewers to approve
  # before the job runs. This is the 'human in the loop' gate.
```

### Pattern 4: Path-Based Monorepo Workflows

```yaml
# Only run service-A tests when service-A changes
on:
  push:
    paths:
      - 'services/service-a/**'
      - '.github/workflows/service-a.yml'

# OR use dorny/paths-filter for dynamic path detection:
jobs:
  changes:
    runs-on: ubuntu-latest
    outputs:
      service-a: ${{ steps.filter.outputs.service-a }}
      service-b: ${{ steps.filter.outputs.service-b }}
    steps:
      - uses: dorny/paths-filter@v3
        id: filter
        with:
          filters: |
            service-a:
              - 'services/service-a/**'
            service-b:
              - 'services/service-b/**'
  
  test-service-a:
    needs: changes
    if: ${{ needs.changes.outputs.service-a == 'true' }}
    # only runs if service-a files changed
```

### 🌍 Where These Patterns Are Seen in Production
- **Shopify:** One reusable workflow for all Ruby microservices — 200+ repos call the same deploy workflow
- **GitHub itself:** Matrix builds across dozens of Ruby versions before releasing a gem
- **Vercel:** Path-based filtering to only rebuild changed Next.js apps in a monorepo
- **Stripe:** Manual approval gates for production deploys — even with 1,000+ automated tests, a human confirms before prod

In [ ]:
"""
Pipeline Orchestration Simulator
================================
Simulates the dependency resolution and parallel execution of a GitHub
Actions workflow. This is how the Actions runner actually works internally.
"""
from __future__ import annotations
import asyncio
import time
import random
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional


class JobStatus(Enum):
    PENDING  = "pending"
    RUNNING  = "running"
    SUCCESS  = "success"
    FAILURE  = "failure"
    SKIPPED  = "skipped"


@dataclass
class Job:
    name: str
    needs: list[str] = field(default_factory=list)
    duration_s: float = 1.0
    fail_probability: float = 0.0
    status: JobStatus = JobStatus.PENDING
    start_time: Optional[float] = None
    end_time: Optional[float] = None

    @property
    def duration_actual(self) -> float:
        if self.start_time and self.end_time:
            return self.end_time - self.start_time
        return 0.0


class WorkflowRunner:
    """Simulates the GitHub Actions job scheduler."""

    def __init__(self, jobs: list[Job]):
        self.jobs = {j.name: j for j in jobs}
        self.start_time = time.monotonic()

    def _all_deps_done(self, job: Job) -> bool:
        return all(
            self.jobs[dep].status in (JobStatus.SUCCESS, JobStatus.FAILURE, JobStatus.SKIPPED)
            for dep in job.needs
        )

    def _any_dep_failed(self, job: Job) -> bool:
        return any(
            self.jobs[dep].status == JobStatus.FAILURE
            for dep in job.needs
        )

    async def _run_job(self, job: Job) -> None:
        job.status = JobStatus.RUNNING
        job.start_time = time.monotonic() - self.start_time
        print(f"  ▶ {job.name:20s} STARTED  (t={job.start_time:.1f}s)")
        await asyncio.sleep(job.duration_s)  # simulate work
        job.end_time = time.monotonic() - self.start_time

        if random.random() < job.fail_probability:
            job.status = JobStatus.FAILURE
            print(f"  ✗ {job.name:20s} FAILED   (t={job.end_time:.1f}s)")
        else:
            job.status = JobStatus.SUCCESS
            print(f"  ✓ {job.name:20s} PASSED   (t={job.end_time:.1f}s, took={job.duration_actual:.1f}s)")

    async def run(self) -> None:
        """Schedule and run jobs respecting dependency order."""
        running: set[asyncio.Task] = set()

        while True:
            # Start eligible jobs
            for job in self.jobs.values():
                if job.status != JobStatus.PENDING:
                    continue
                if not self._all_deps_done(job):
                    continue
                if self._any_dep_failed(job):
                    job.status = JobStatus.SKIPPED
                    print(f"  ⊘ {job.name:20s} SKIPPED  (upstream failed)")
                    continue
                task = asyncio.create_task(self._run_job(job))
                running.add(task)
                task.add_done_callback(running.discard)

            # Check if all done
            all_done = all(
                j.status != JobStatus.PENDING and j.status != JobStatus.RUNNING
                for j in self.jobs.values()
            )
            if all_done and not running:
                break

            await asyncio.sleep(0.1)  # scheduler tick

        total = time.monotonic() - self.start_time
        sequential = sum(j.duration_s for j in self.jobs.values() if j.status != JobStatus.SKIPPED)
        print(f"\n  Total wall time:     {total:.1f}s")
        print(f"  Sequential time:     {sequential:.1f}s")
        print(f"  Parallelism savings: {sequential - total:.1f}s ({(1 - total/sequential)*100:.0f}% faster)")

In [ ]:
# Run the production workflow simulation
random.seed(0)

jobs = [
    Job("lint",             needs=[],                          duration_s=1.5),
    Job("unit-tests",       needs=[],                          duration_s=3.0),
    Job("security-scan",    needs=[],                          duration_s=2.0),
    Job("build",            needs=["lint","unit-tests","security-scan"], duration_s=4.0),
    Job("integration-tests",needs=["build"],                  duration_s=5.0),
    Job("deploy-staging",   needs=["integration-tests"],       duration_s=2.0),
    Job("deploy-production",needs=["deploy-staging"],          duration_s=2.0),
]

print("=== Pipeline Execution Simulation ===")
print(f"  {len(jobs)} jobs | Gate 1 runs in parallel\n")
await WorkflowRunner(jobs).run()

---
## 7 · Monorepo vs Polyrepo

### 🧠 Mental Model — *The Organization Is the Architecture*

> **Conway's Law: organizations design systems that mirror their communication structure. Monorepos enforce cross-team collaboration because changes to shared code are visible to all teams. Polyrepos enforce autonomy because teams own their entire stack independently. Choose the structure that matches the organizational model you want.**

### Side-by-Side Comparison

| Dimension | Monorepo | Polyrepo |
|---|---|---|
| **Code sharing** | Trivial (same repo, same import) | Hard (publish packages, manage versions) |
| **Atomic cross-service changes** | One PR changes 5 services at once | Requires 5 PRs + coordination |
| **CI scope** | Must scope tests to changed paths | Each repo runs its own full suite |
| **Dependency management** | One lockfile, unified versions | Each repo manages its own |
| **Build tooling** | Requires Nx, Turborepo, or Bazel | Standard tools work fine |
| **Team autonomy** | Lower — shared ownership of repo | Higher — teams own their repo |
| **Discoverability** | Everything in one place | Need to know where to look |
| **Blast radius of changes** | Can accidentally break others | Contained to your repo |

### 🌍 Real-World Choices

| Company | Choice | Why |
|---|---|---|
| **Google** | Monorepo (Piper, 2B+ LOC) | Atomic cross-team refactoring; everyone sees everyone |
| **Facebook/Meta** | Monorepo (Mercurial + Eden) | Same rationale; tools at scale |
| **Twitter/X** | Monorepo → Polyrepo migration | Growing pains with tooling |
| **Netflix** | Polyrepo | Each microservice team fully autonomous |
| **Shopify** | Monorepo (Rails Rails) | Core platform is one repo; tooling around it |
| **Vercel** | Monorepo (Turborepo) | They built the tool, they use it |

### The CI Implication: Monorepo Requires Smarter Pipelines

```yaml
# Naive monorepo CI: runs everything every time (slow, expensive)
on: push
jobs:
  test-all-services:
    run: pytest services/*/tests/  # All 30 services, every commit

# Smart monorepo CI: affected analysis (only changed services)
# Tools: Nx, Turborepo, Bazel, or manual path-filter
jobs:
  detect-changes:
    outputs:
      services: ${{ steps.changed.outputs.services }}
    steps:
      - id: changed
        run: |
          # nx affected --target=test outputs only changed service names
          echo "services=$(nx show projects --affected)" >> $GITHUB_OUTPUT

  test:
    needs: detect-changes
    strategy:
      matrix:
        service: ${{ fromJson(needs.detect-changes.outputs.services) }}
    steps:
      - run: pytest services/${{ matrix.service }}/tests/
```

---
## 8 · Real-World Case Studies

### Netflix: The Deployment Freedom Model

**Problem:** 2,000+ microservices, 500+ engineers, multiple deployments per hour.  
**Solution:** Spinnaker (built by Netflix, now open source) — a pipeline orchestration platform.

**Key decisions:**
- Each microservice team owns their pipeline completely
- Canary deployments by default: 1% → 10% → 50% → 100% with automated metric gates
- Automated rollback: if `error_rate > 0.1%` for 5 minutes at any canary stage → auto-rollback
- **Chaos Engineering** (Chaos Monkey): pipelines must include chaos tests

```
Netflix Deploy Flow:
Code pushed → Nebula build (Gradle) → Docker image → Spinnaker pipeline
  → Bake AMI (or push image) → Deploy canary (1%) → Wait 15min
  → Compare metrics (p99 latency, error rate) vs baseline
  → Auto-promote or auto-rollback based on statistical significance
  → Blue/green swap to 100%
```

### Shopify: The Sub-10-Minute Pipeline

**Problem:** 350+ engineers pushing to a Rails monolith, must deploy fast without breaking the platform.  
**Solution:** Ruthless pipeline optimization.

**Key decisions:**
- Parallel test sharding: 10,000+ Ruby tests split across 20 parallel workers
- Merging is restricted: CI must be green before any human can merge
- Feature flags (Flipper): all features behind flags → trunk-based development
- Custom deployment platform: Shipit — not GitHub Actions, because they needed fine-grained control
- **Result:** Code change → production in < 10 minutes

### Google: Trunk-Based at 25,000 Engineers

**Problem:** 25,000+ engineers in one repo, billions of lines of code.  
**Solution:** Bazel (hermetic builds) + custom CI (TAP) + automatic large-scale changes (LSC).

**Key decisions:**
- Every commit to trunk triggers tests for everything that *depends* on the changed code
- Bazel's hermetic builds mean the same binary is produced regardless of machine
- "Large-Scale Changes" tooling: automated PRs that modify 10,000+ files for global refactors
- **No branches:** everything on trunk, behind feature flags
- **Result:** 1 billion lines changed per year, safely

---
## 9 · The DevOps Architect's Pipeline Design Framework

### 🧠 Mental Model — *The 7-Question Design Review*

> **Before you write a single line of pipeline YAML, answer these 7 questions. They constrain every design decision downstream.**

```
1. CLARIFY   → What is the deployment target? (K8s, ECS, serverless, VMs)
               What is the deployment frequency goal? (daily, hourly, on-demand)
               What are the compliance requirements? (SOC2, HIPAA, PCI-DSS)
               
2. ARTIFACT  → What is the deployable artifact? (Docker image, ZIP, JAR, Helm chart)
               Where is it stored? (GHCR, ECR, Artifactory, S3)
               
3. GATES     → What quality gates are required before production?
               Who has authority to approve production deploys?
               
4. SECRETS   → What credentials does the pipeline need?
               Can OIDC eliminate long-lived credentials?
               
5. SPEED     → What is the acceptable feedback time for a PR? (target: <10 min)
               What can be parallelized? What can be cached?
               
6. FAILURE   → What happens when a deploy fails? (auto-rollback? manual? alert?)
               How do you know when it fails? (metrics, synthetic tests, alerts)
               
7. OPERATE   → Who maintains the pipeline? (pipeline is code — it has an owner)
               How do you handle flaky tests? (fix them — never "just re-run")
               How do you track pipeline health? (failure rate, mean time to green)
```

### The Senior vs Junior Answer

**Junior:** "I'll set up GitHub Actions to run tests and deploy on push to main."

**Senior:** "Let me clarify the requirements first. What's the deployment target and frequency? What are the compliance constraints? Given a K8s target with 10+ deploys/day, I'd recommend: parallel Gate 1 (lint, unit tests, SAST) → Gate 2 (build + image scan) → Gate 3 (staging deploy + smoke tests) → Gate 4 (canary prod deploy via Argo Rollouts). OIDC for all cloud credentials. Path filtering for monorepo scope. Concurrency groups to prevent deployment races. Mean time to deploy target: under 8 minutes for most changes."

---

### 📚 What to Study Next

1. **Module 02 — Jenkins:** The same pipeline concepts but in enterprise context: Groovy DSL, shared libraries, multi-branch pipelines, and why Jenkins still dominates in regulated industries
2. **Module 03 — Docker:** The artifact created by your pipeline. Understanding Docker layers and multi-stage builds makes your build stage 2-5× faster
3. **Module 04 — Kubernetes:** The deployment target. Understanding how K8s deployments work changes how you write your deploy stage
4. **`examples/` folder:** Deep-dive notebooks on Git internals, branching, and advanced Actions patterns